# 🎙️ Fine-tuning Whisper Small — Malagasy STT

**Objectif :** Fine-tuner `openai/whisper-small` pour la transcription vocale en langue Malagasy.

**Sources de données :**
- `converted_dataset.zip` → fichiers audio WAV propres (`wavs/`)
- `dataset_malagasy_tts.zip` → `metadata.csv` avec les transcriptions

**Format metadata.csv (séparateur `|`) :**
```
wavs/AT-01-Genesisy-ch_01-T0013015.wav|ary aizina no tambonin'ny lalina.
```

**Sauvegarde :** Google Drive — `ML Malagasy_TTS/`

---
> ⚠️ **Avant de commencer :** Activer le GPU dans Colab → `Exécution > Modifier le type d'exécution > GPU (T4)`

## 📦 Cellule 1 — Installation des dépendances

In [ ]:
!pip install -q --upgrade \
    datasets \
    transformers \
    accelerate \
    evaluate \
    jiwer \
    librosa \
    soundfile \
    tensorboard

print('✅ Dépendances installées')

## 📂 Cellule 2 — Montage Google Drive et extraction des datasets

In [ ]:
from google.colab import drive
import os, zipfile, shutil

# --- Montage Drive ---
drive.mount('/content/drive')

# --- Chemins dans Drive ---
DRIVE_BASE     = '/content/drive/MyDrive/ML Malagasy_TTS/Dataset'
AUDIO_ZIP      = os.path.join(DRIVE_BASE, 'converted_dataset.zip')
META_ZIP       = os.path.join(DRIVE_BASE, 'dataset_malagasy_tts.zip')

# --- Dossiers de travail locaux (plus rapide que Drive) ---
WORK_DIR       = '/content/malagasy_whisper'
AUDIO_DIR      = os.path.join(WORK_DIR, 'wavs')
META_DIR       = os.path.join(WORK_DIR, 'meta')
OUTPUT_DIR     = '/content/drive/MyDrive/ML Malagasy_TTS/whisper-small-mg'

os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(META_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- Extraction des audios ---
print('📦 Extraction des fichiers audio...')
with zipfile.ZipFile(AUDIO_ZIP, 'r') as z:
    z.extractall(WORK_DIR)
print(f'✅ Audio extrait dans : {WORK_DIR}')

# --- Extraction du metadata.csv ---
print('📦 Extraction de metadata.csv...')
with zipfile.ZipFile(META_ZIP, 'r') as z:
    # On extrait uniquement metadata.csv
    members = [m for m in z.namelist() if 'metadata.csv' in m]
    if not members:
        raise FileNotFoundError('metadata.csv introuvable dans dataset_malagasy_tts.zip')
    z.extract(members[0], META_DIR)
    META_CSV = os.path.join(META_DIR, members[0])
print(f'✅ metadata.csv extrait : {META_CSV}')

## 🔍 Cellule 3 — Vérification des données

In [ ]:
import pandas as pd

# --- Lecture metadata.csv (séparateur pipe) ---
df = pd.read_csv(META_CSV, sep='|', header=None, names=['audio_path', 'transcription'])
df['audio_path'] = df['audio_path'].str.strip()
df['transcription'] = df['transcription'].str.strip()

print(f'📊 Total échantillons : {len(df)}')
print(df.head(5))

# --- Vérification que les fichiers audio existent bien ---
def check_audio(row):
    # Le chemin dans metadata est relatif (wavs/xxx.wav)
    # On le résout par rapport au WORK_DIR
    full_path = os.path.join(WORK_DIR, row['audio_path'])
    return os.path.exists(full_path)

df['exists'] = df.apply(check_audio, axis=1)
missing = df[~df['exists']]
print(f'\n⚠️  Fichiers manquants : {len(missing)}')
if len(missing) > 0:
    print(missing['audio_path'].head(5))

# Garder uniquement les fichiers existants
df = df[df['exists']].reset_index(drop=True)
df['full_path'] = df['audio_path'].apply(lambda p: os.path.join(WORK_DIR, p))
print(f'\n✅ Échantillons utilisables : {len(df)}')

## ✂️ Cellule 4 — Séparation Train / Test

In [ ]:
from sklearn.model_selection import train_test_split

TRAIN_RATIO = 0.90  # 90% train, 10% test

df_train, df_test = train_test_split(df, test_size=1 - TRAIN_RATIO, random_state=42)
df_train = df_train.reset_index(drop=True)
df_test  = df_test.reset_index(drop=True)

print(f'🏋️  Entraînement : {len(df_train)} échantillons')
print(f'🧪 Test          : {len(df_test)} échantillons')

## 🤗 Cellule 5 — Construction du Dataset Hugging Face

In [ ]:
import librosa
import numpy as np
from datasets import Dataset, DatasetDict, Audio

TARGET_SR = 16000  # Whisper attend du 16 kHz

def df_to_hf_dataset(dataframe):
    return Dataset.from_dict({
        'audio': dataframe['full_path'].tolist(),
        'sentence': dataframe['transcription'].tolist(),
    }).cast_column('audio', Audio(sampling_rate=TARGET_SR))

dataset = DatasetDict({
    'train': df_to_hf_dataset(df_train),
    'test':  df_to_hf_dataset(df_test),
})

print(dataset)
print('\n🔊 Exemple :', dataset['train'][0]['sentence'])

## 🧠 Cellule 6 — Chargement du modèle Whisper Small

In [ ]:
from transformers import (
    WhisperFeatureExtractor,
    WhisperTokenizer,
    WhisperProcessor,
    WhisperForConditionalGeneration,
)

MODEL_NAME = 'openai/whisper-small'
LANGUAGE   = 'Malagasy'
TASK       = 'transcribe'

feature_extractor = WhisperFeatureExtractor.from_pretrained(MODEL_NAME)
tokenizer         = WhisperTokenizer.from_pretrained(MODEL_NAME, language=LANGUAGE, task=TASK)
processor         = WhisperProcessor.from_pretrained(MODEL_NAME, language=LANGUAGE, task=TASK)
model             = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)

# Désactiver la détection automatique de langue (on force le Malagasy)
model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(language=LANGUAGE, task=TASK)
model.config.suppress_tokens    = []

print(f'✅ Modèle {MODEL_NAME} chargé')
print(f'   Paramètres : {model.num_parameters():,}')

## ⚙️ Cellule 7 — Préparation des features (feature extraction)

In [ ]:
def prepare_dataset(batch):
    audio = batch['audio']
    # Extraire le log-Mel spectrogram
    batch['input_features'] = feature_extractor(
        audio['array'],
        sampling_rate=audio['sampling_rate']
    ).input_features[0]
    # Encoder la transcription en token IDs
    batch['labels'] = tokenizer(batch['sentence']).input_ids
    return batch

print('🔄 Préparation du dataset train...')
dataset['train'] = dataset['train'].map(
    prepare_dataset,
    remove_columns=dataset.column_names['train'],
    num_proc=1
)

print('🔄 Préparation du dataset test...')
dataset['test'] = dataset['test'].map(
    prepare_dataset,
    remove_columns=dataset.column_names['test'],
    num_proc=1
)

print('✅ Features prêtes')
print(dataset)

## 🗂️ Cellule 8 — Data Collator

In [ ]:
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # Padding des input_features
        input_features = [{'input_features': f['input_features']} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors='pt')

        # Padding des labels
        label_features = [{'input_ids': f['labels']} for f in features]
        labels_batch   = self.processor.tokenizer.pad(label_features, return_tensors='pt')

        # Remplacer le padding par -100 (ignoré par la loss)
        labels = labels_batch['input_ids'].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        # Supprimer le token BOS si présent en début de labels
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch['labels'] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)
print('✅ Data collator prêt')

## 📏 Cellule 9 — Métrique d'évaluation (WER)

In [ ]:
import evaluate

metric = evaluate.load('wer')

def compute_metrics(pred):
    pred_ids  = pred.predictions
    label_ids = pred.label_ids

    # Remplacer -100 par pad_token_id pour le décodage
    label_ids[label_ids == -100] = tokenizer.pad_token_id

    pred_str  = tokenizer.batch_decode(pred_ids,  skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer_score = 100 * metric.compute(predictions=pred_str, references=label_str)
    return {'wer': round(wer_score, 2)}

print('✅ Métrique WER prête')

## 🏋️ Cellule 10 — Configuration de l'entraînement

In [ ]:
from transformers import Seq2SeqTrainingArguments

# -------------------------------------------------------
# 📌 Paramètres ajustables selon la taille de ton dataset
# -------------------------------------------------------
MAX_STEPS   = 4000   # Augmenter si dataset > 10h
EVAL_STEPS  = 500    # Évaluation toutes les N étapes
SAVE_STEPS  = 500    # Sauvegarde checkpoint toutes les N étapes
BATCH_SIZE  = 16     # Réduire à 8 si OOM (Out Of Memory)
GRAD_ACCUM  = 1      # Augmenter si batch réduit (ex: BATCH=8, ACCUM=2)
LR          = 1e-5   # Learning rate recommandé pour whisper-small
WARMUP      = 500
# -------------------------------------------------------

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_steps=WARMUP,
    max_steps=MAX_STEPS,
    gradient_checkpointing=True,
    fp16=True,
    evaluation_strategy='steps',
    save_strategy='steps',
    eval_steps=EVAL_STEPS,
    save_steps=SAVE_STEPS,
    logging_steps=50,
    predict_with_generate=True,
    generation_max_length=225,
    load_best_model_at_end=True,
    metric_for_best_model='wer',
    greater_is_better=False,
    report_to=['tensorboard'],
    save_total_limit=3,       # Garder seulement les 3 derniers checkpoints
)

print('✅ Arguments d\'entraînement configurés')
print(f'   Steps total  : {MAX_STEPS}')
print(f'   Batch size   : {BATCH_SIZE}')
print(f'   Learning rate: {LR}')
print(f'   Sauvegarde   : {OUTPUT_DIR}')

## 🚀 Cellule 11 — Lancement de l'entraînement

In [ ]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=dataset['train'],
    eval_dataset=dataset['test'],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=processor.feature_extractor,
)

print('🚀 Début de l\'entraînement...')
trainer.train()
print('\n🎉 Entraînement terminé !')

## 💾 Cellule 12 — Sauvegarde du modèle final sur Drive

In [ ]:
FINAL_MODEL_DIR = os.path.join(OUTPUT_DIR, 'final_model')
os.makedirs(FINAL_MODEL_DIR, exist_ok=True)

trainer.save_model(FINAL_MODEL_DIR)
processor.save_pretrained(FINAL_MODEL_DIR)

print(f'✅ Modèle final sauvegardé dans : {FINAL_MODEL_DIR}')

## 📊 Cellule 13 — Évaluation finale sur le set de test

In [ ]:
results = trainer.evaluate()
print('\n📊 Résultats finaux :')
for k, v in results.items():
    print(f'   {k}: {v}')

## 🎤 Cellule 14 — Test d'inférence sur un fichier audio

In [ ]:
import librosa
from transformers import pipeline

# Charger le modèle fine-tuné depuis Drive
asr_pipeline = pipeline(
    'automatic-speech-recognition',
    model=FINAL_MODEL_DIR,
    device=0  # GPU
)

# --- Tester sur un fichier du dataset de test ---
test_sample    = df_test.iloc[0]
audio_path     = test_sample['full_path']
reference_text = test_sample['transcription']

result = asr_pipeline(audio_path)

print(f'🎵 Fichier     : {os.path.basename(audio_path)}')
print(f'📝 Référence   : {reference_text}')
print(f'🤖 Prédiction  : {result["text"]}')

## 📈 Cellule 15 — Visualiser les logs avec TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {OUTPUT_DIR}

---
## 📌 Résumé des fichiers générés

| Fichier / Dossier | Description |
|---|---|
| `whisper-small-mg/final_model/` | Modèle fine-tuné final |
| `whisper-small-mg/checkpoint-*/` | Checkpoints intermédiaires |
| `whisper-small-mg/runs/` | Logs TensorBoard |

## 💡 Conseils si problèmes

- **OOM (Out Of Memory)** → Réduire `BATCH_SIZE` à `8` et mettre `GRAD_ACCUM = 2`
- **WER ne baisse pas** → Vérifier la qualité audio (16kHz, mono) et augmenter `MAX_STEPS`
- **Déconnexion Colab** → Les checkpoints sont sur Drive, relancer depuis `trainer.train(resume_from_checkpoint=True)`
- **Langue non reconnue** → Le Malagasy (`mg`) est supporté par Whisper multilingual small ✅